# YOLO26n-seg stool model, 3-group variant: constipated / normal / loose

**Before running:** Runtime → Change runtime type → GPU → Save. With compute units, pick **A100** if offered, else **L4**, else T4. This model is small, so a full run costs only a few units on any of them. Then run cells top to bottom.

Same pipeline as v3, but the 7 Bristol types are merged into 3 classes: **0 = constipated (types 1-2), 1 = normal (3-5), 2 = loose (6-7)**.
The 7-class detector plateaued at ~0.65 val mAP50 because adjacent types are near-indistinguishable at this data size; merging removes that confusion.
Everything else (clean 640 px data, Drive saving, resume cell, ONNX export) is unchanged. Expect ~20 min on an A100.

Dataset: "stool classification" by *smartgut* on [Roboflow Universe](https://universe.roboflow.com/smartgut/stool-classification-uvnqx), CC BY 4.0. Not medical advice; see `DATASET_ATTRIBUTION.md`.

In [ ]:
# 1) GPU check. Prints which GPU you got (T4 / L4 / A100).
!nvidia-smi -L

In [ ]:
# 2) Install + mount Drive. Approve the Drive popup.
%pip install -q -U ultralytics onnx onnxruntime onnxslim
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/stool_yolo'   # all outputs live here
RUN   = f'{DRIVE}/runs/stool_3group'              # this training run
os.makedirs(DRIVE, exist_ok=True)
import ultralytics; ultralytics.checks()

## 3) Dataset

Upload `stool_yolo_seg_v3_640_3group.zip` **once**. It's copied to Drive, so after a disconnect this cell finds it there and skips the upload.

In [ ]:
import shutil, zipfile, glob
zip_in_drive = f'{DRIVE}/stool_yolo_seg_v3_640_3group.zip'
if not os.path.exists(zip_in_drive):
    from google.colab import files
    uploaded = files.upload()                       # choose stool_yolo_seg_v3_640_3group.zip
    shutil.copy(next(iter(uploaded)), zip_in_drive)
    print('copied zip to Drive')
else:
    print('found zip in Drive, skipping upload')

DATA = '/content/dataset'                          # local disk = fast training
shutil.rmtree(DATA, ignore_errors=True); os.makedirs(DATA)
zipfile.ZipFile(zip_in_drive).extractall(DATA)
!sed -i 's|^path: .*|path: /content/dataset|' /content/dataset/data.yaml
for s in ['train', 'valid', 'test']:
    print(s, len(glob.glob(f'{DATA}/images/{s}/*')), 'images')

## 4) Oversample the rare class

Bristol **type 2** (class id `1`) is the rarest class (recall 0.40 in v1).
Every training image containing a type-2 polygon gets **2 extra copies**. Validation and test are untouched, so metrics stay honest.

In [ ]:
from pathlib import Path
OVERSAMPLE = {}   # 3 groups are already balanced (~475 / 637 / 538 polygons); nothing to oversample

lbl_dir, img_dir = Path(DATA)/'labels/train', Path(DATA)/'images/train'
added = 0
for lbl in sorted(lbl_dir.glob('*.txt')):
    if lbl.stem.startswith('dup'): continue
    classes = {int(l.split()[0]) for l in lbl.read_text().splitlines() if l.strip()}
    extra = max([OVERSAMPLE.get(c, 0) for c in classes] or [0])
    if not extra: continue
    img = next(img_dir.glob(lbl.stem + '.*'))
    for k in range(extra):
        shutil.copy(img, img_dir / f'dup{k}_{img.name}')
        shutil.copy(lbl, lbl_dir / f'dup{k}_{lbl.name}')
        added += 1
print(f'added {added} duplicate images; train now has {len(list(img_dir.glob("*")))}')

## 5) Train (640 px, 200 epochs)

Knobs, so you know what's what:
- `cos_lr=True`: learning rate decays on a cosine curve, usually a point or two better than linear on long runs.
- `patience=50`: only stops if validation hasn't improved for 50 epochs. Long runs have long flat stretches that then move again.
- `close_mosaic=30`: last 30 epochs train on clean, un-mosaiced images so the model settles on realistic inputs.
- `copy_paste=0.3`: pastes masked stool instances into other images. Segmentation-only trick, strong on small datasets.
- `flipud=0.5, degrees=15`: stool has no "up", so vertical flips and rotations are free data.
- `mixup=0.1`, `hsv_*`: blends images and jitters colour, which fights the model memorising lighting/toilet colours.
- `optimizer='auto'`: with ~5k total steps Ultralytics picks AdamW and sets the learning rate itself. Setting `lr0` would be silently ignored, so it isn't.

Watch `Mask ... mAP50`. If Colab disconnects, go to **cell 9** instead of re-running this one.

In [ ]:
from ultralytics import YOLO

MODEL = 'yolo26n-seg.pt'   # keep nano for the browser. 'yolo26s-seg.pt' = ~+3 mAP, 4x bigger file, ~3x slower on phone.

model = YOLO(MODEL)
results = model.train(
    data=f'{DATA}/data.yaml',
    epochs=200, patience=50, imgsz=640, batch=32,
    cache='ram', workers=8,   # dataset is ~1 GB decoded; caching removes the CPU bottleneck so the GPU is actually used
    project=f'{DRIVE}/runs', name='stool_3group', exist_ok=True,
    cos_lr=True, close_mosaic=30, warmup_epochs=5,
    optimizer='auto',   # = AdamW with lr ~0.0009 for a run this short; lr0 is ignored in auto mode
    # augmentation
    copy_paste=0.3, mixup=0.1, flipud=0.5, fliplr=0.5, degrees=15, translate=0.1, scale=0.5,
    hsv_h=0.02, hsv_s=0.6, hsv_v=0.4,
    plots=True,
)

In [ ]:
# 6) Evaluate best.pt on validation and test, show curves + confusion matrix.
from IPython.display import Image, display
print('files in run folder:', sorted(os.listdir(RUN)))

best = YOLO(f'{RUN}/weights/best.pt')
val_m  = best.val(data=f'{DATA}/data.yaml', split='val',  imgsz=640, plots=False)
test_m = best.val(data=f'{DATA}/data.yaml', split='test', imgsz=640, plots=False)
print(f'VAL  box mAP50 {val_m.box.map50:.3f}  mask mAP50 {val_m.seg.map50:.3f}  mask mAP50-95 {val_m.seg.map:.3f}')
print(f'TEST box mAP50 {test_m.box.map50:.3f}  mask mAP50 {test_m.seg.map50:.3f}  mask mAP50-95 {test_m.seg.map:.3f}')

for img in ['results.png', 'confusion_matrix_normalized.png', 'MaskF1_curve.png', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    p = f'{RUN}/{img}'
    print(img) if os.path.exists(p) else print('missing:', p)
    if os.path.exists(p): display(Image(p, width=900))

## 7) Export ONNX (browser) + verify shapes

`nms=False` selects YOLO26's end-to-end head, so the ONNX outputs final detections (no NMS in JavaScript).
Input `[1,3,640,640]` float32; expected outputs `[1,300,38]` (x1,y1,x2,y2,score,class 0-2, 32 mask coeffs) and `[1,32,160,160]` (mask prototypes). In the browser, resize the photo to 640×640 and divide box coords by 4 to index the prototype grid.

In [ ]:
onnx_path = best.export(format='onnx', imgsz=640, nms=False, dynamic=False, half=False, simplify=True, opset=17)
shutil.copy(onnx_path, f'{DRIVE}/stool_3group.onnx')
shutil.copy(f'{RUN}/weights/best.pt', f'{DRIVE}/stool_3group.pt')
print('saved to Drive:', f'{DRIVE}/stool_3group.onnx', f'{DRIVE}/stool_3group.pt')

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
for i in sess.get_inputs():  print('INPUT ', i.name, i.shape, i.type)
for o in sess.get_outputs(): print('OUTPUT', o.name, o.shape, o.type)

In [ ]:
# 8) Download to your computer (they're also in Drive > stool_yolo/).
from google.colab import files
files.download(f'{DRIVE}/stool_3group.pt')
files.download(f'{DRIVE}/stool_3group.onnx')

## Optional: let Ultralytics search the hyperparameters

Only if you have ~6 hours of Colab left (at 640 px each short run is ~4x slower than at 320; consider `imgsz=320` for the search and 640 for the final run) and want the last 1 to 3 mAP points. This runs 20 short trainings (30 epochs each), mutating learning rate, augmentation strengths, loss weights etc., keeps the best set,
then does the full 200-epoch run with those values. Run it **instead of** cell 5, after cells 1 to 4.
The optimizer is fixed to AdamW here (not `auto`) so that the tuned learning rate actually takes effect.

In [ ]:
from ultralytics import YOLO
tuner_model = YOLO('yolo26n-seg.pt')
tuner_model.tune(
    data=f'{DATA}/data.yaml', imgsz=640, batch=32,
    epochs=30, iterations=20, patience=10,
    optimizer='AdamW', lr0=0.001,   # explicit optimizer so lr0 mutations are not ignored; 0.001 is the right ballpark for AdamW
    project=f'{DRIVE}/runs', name='tune', exist_ok=True,
    plots=False, save=False, val=True,
)
# best values land in: {DRIVE}/runs/tune/best_hyperparameters.yaml
import yaml, glob
best_hp = yaml.safe_load(open(glob.glob(f'{DRIVE}/runs/tune*/best_hyperparameters.yaml')[-1]))
print(best_hp)

# Final long run with the tuned values (same as cell 5 otherwise)
model = YOLO('yolo26n-seg.pt')
results = model.train(
    data=f'{DATA}/data.yaml', epochs=200, patience=50, imgsz=640, batch=32, cache='ram', workers=8,
    project=f'{DRIVE}/runs', name='stool_3group', exist_ok=True,
    cos_lr=True, close_mosaic=30, plots=True, optimizer='AdamW',
    **best_hp,
)

## 9) Resume after a disconnect (only if needed)

If Colab died mid-training: run cells **1, 2, 3, 4** again (the zip comes from Drive, no re-upload), then run this cell instead of cell 5.
It picks up from the last saved epoch with the same settings. Then continue with cells 6 to 8.

In [ ]:
from ultralytics import YOLO
model = YOLO(f'{RUN}/weights/last.pt')
results = model.train(resume=True)

## Going further (if you still need more)

- **Two-stage pipeline**: a 1-class "stool" seg model (finds and masks it, will score 0.9+ mAP50) plus a `yolo26n-cls` classifier run on the cropped stool at 224 px to pick the Bristol type. Classification on a tight crop sees far more detail than a detector head on the full frame, and the classifier can be trained with label smoothing across neighbouring types. Both models together are still under 15 MB.
- **Merge to 3 groups** (constipated 1 to 2, normal 3 to 5, loose 6 to 7): typically pushes mAP50 past 0.85 because the adjacent-type confusion disappears. Change the class ids in the label files before training.
- **More type 2 and type 4 photos** in Roboflow and re-export. Real data beats every trick above.
- **Small model** (`MODEL = 'yolo26s-seg.pt'`) if you decide single-photo latency is fine and 40 MB is an acceptable download.
- **App threshold**: v1 had high precision, low recall. Use `score > 0.2` in the browser instead of 0.25 to catch more instances.